# ARTI 402 — Deep Learning
## Lab 4 — Recurrent Networks: RNN, LSTM and GRU

**Week 4 · Neural Network Architectures (cont.)**
Recurrent neural networks (RNNs) for sequential data; Long Short-Term Memory (LSTM) and Gated
Recurrent Units (GRUs).

| | |
|---|---|
| **Marks** | **1 mark** (graded) |
| **Estimated time** | 100–120 minutes |
| **Prerequisites** | Labs 1–3 — `Layer_Dense`, softmax, `train_head`, the idea of weight sharing |



| **Reading** | Chris Olah — [*Understanding LSTM Networks*](http://colah.github.io/posts/2015-08-Understanding-LSTMs/) | 20 min | The classic. Most LSTM diagrams online come from here |
| *Optional, advanced* | Distill — [*Visualizing Memorization in RNNs*](https://distill.pub/2019/memorization-in-rnns/) | — | Interactive: see which earlier words an LSTM or GRU actually uses |

---

### How to work through the notebook

* Sections are labelled **Idea** (read and run), **Exercise** (you write code) and **Checkpoint** (a short answer).
* Every exercise cell is marked `# TODO`. Do not delete the cells above it — later cells depend on them.
* Run cells **in order**, top to bottom. If something breaks, restart the kernel and run all.
* The graded **Assessment** is at the very end.

---

### The one idea to hold onto

In Lab 3, a CNN used **one filter everywhere in space** — the same nine weights slid across the
whole image. That is why it could spot a shape anywhere.

An RNN uses **one cell everywhere in time** — the same weights applied at every step of a
sequence. Same trick, different axis.

We work in four stages again: **see it → watch it → build it → prove it.**

### By the end of this lab you should be able to

1. Explain why a dense network cannot handle word order or variable-length input.
2. Implement an RNN cell and unroll it over a sequence.
3. Count recurrent parameters, and explain why they do not depend on sequence length.
4. Show why a plain RNN forgets, using the idea of repeated multiplication.
5. Implement an LSTM cell and explain its three stages and its two kinds of memory.
6. Compare RNN, LSTM and GRU, and show experimentally that the LSTM remembers longer.


---
## Setup

### Files you need

Put this file in the **same folder** as the notebook:

| File | What it is |
|---|---|
| `arti402_figures.py` | draws the diagrams (this is the updated version — it replaces Lab 3's) |

### The code

The next cell carries forward Labs 2–3 and adds the weights and data for this lab. Read it, run it,
move on.


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

D = 4      # features per time step
H = 12     # size of the hidden state (the memory)


# ---------- carried forward from Labs 2-3 ----------

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)


def categorical_crossentropy(y_pred, y_true):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return np.mean(-np.log(y_pred[range(len(y_pred)), y_true]))


class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.1 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases
        return self.output


def numerical_gradient(param, loss_fn, h=1e-4):
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        original = param[idx]
        param[idx] = original + h; loss_plus = loss_fn()
        param[idx] = original - h; loss_minus = loss_fn()
        param[idx] = original
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        it.iternext()
    return grad


def train_head(X, y, n_classes=3, epochs=120, learning_rate=0.5, seed=42):
    """Train one dense layer + softmax on X. The Lab 2 loop, packaged up."""
    np.random.seed(seed)
    head = Layer_Dense(X.shape[1], n_classes)
    loss_fn = lambda: categorical_crossentropy(softmax(head.forward(X)), y)
    params = [head.weights, head.biases]
    for _ in range(epochs):
        grads = [numerical_gradient(p, loss_fn) for p in params]
        for p, g in zip(params, grads):
            p -= learning_rate * g
    return head


def accuracy(head, X, y):
    return np.mean(np.argmax(softmax(head.forward(X)), axis=1) == y)


# ---------- weights for this lab ----------

def make_rnn_weights(seed=0):
    """Small random RNN weights: (Wx, Wh, b)."""
    r = np.random.RandomState(seed)
    Wx = r.randn(D, H) * 0.5
    Wh = r.randn(H, H) / np.sqrt(H) * 0.9
    b = np.zeros(H)
    return Wx, Wh, b


def make_lstm_weights(forget_bias=4.0, input_bias=-2.0, seed=0):
    """LSTM weights as a dict: gate name -> (Wx, Wh, b).

    f = forget gate, i = input gate, g = candidate memory, o = output gate.
    """
    r = np.random.RandomState(seed)
    W = {}
    for name in ["f", "i", "g", "o"]:
        Wx = r.randn(D, H) * 0.5
        Wh = r.randn(H, H) / np.sqrt(H) * 0.9
        W[name] = (Wx, Wh, np.zeros(H))
    W["f"] = (W["f"][0], W["f"][1], W["f"][2] + forget_bias)
    W["i"] = (W["i"][0], W["i"][1], W["i"][2] + input_bias)
    return W


# ---------- the data for this lab ----------

def make_sequences(n_per_class, T, seed, noise=0.3):
    """Sequences of T steps, D features each.

    Step 1 carries a signal: a spike of 3.0 in feature 0, 1 or 2 -> the class.
    Steps 2..T are random noise.  To classify, a model must REMEMBER step 1.
    """
    rng = np.random.RandomState(seed)
    X, y = [], []
    for k in range(3):
        for _ in range(n_per_class):
            s = rng.randn(T, D) * noise
            s[0] = 0.0
            s[0, k] = 3.0
            X.append(s)
            y.append(k)
    return np.array(X), np.array(y)


print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("\nSetup OK")

In [ ]:
# Diagrams live in arti402_figures.py, in the same folder as this notebook.
from arti402_figures import (draw_unrolled_rnn, draw_repeated_multiplication,
                             draw_lstm_cell, draw_recurrent_params,
                             draw_lab4_pipeline)

print("figure helpers loaded")


---
# Stage 1 · See it work

## Idea 1 — Networks that read and write

### Do this now (10 minutes)

Open Andrej Karpathy's **[The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)**
and scroll through the examples. Don't read the maths — just look at what comes out.

A small recurrent network, trained on nothing but raw text **one character at a time**, learns to
produce text that looks like Shakespeare, like Wikipedia markup, like LaTeX, and even like C code
from the Linux kernel. Nobody told it what a word is, or what a bracket is for.

Notice what it must be doing. To close a bracket correctly it has to **remember** that one was
opened, possibly many characters earlier. That memory is what this lab is about.

### Why a dense network cannot do this

Two problems, and you can demonstrate the first one in three lines.

**Problem 1 — order is lost.** The simplest way to feed a sentence to a dense network is a
"bag of words": count how often each word appears. Run the cell.


In [ ]:
vocab = ["dog", "bites", "man"]

def bag_of_words(sentence):
    words = sentence.split()
    return np.array([words.count(w) for w in vocab])

a = bag_of_words("dog bites man")
b = bag_of_words("man bites dog")

print("'dog bites man' ->", a)
print("'man bites dog' ->", b)
print("identical inputs?", np.array_equal(a, b))

Two sentences with opposite meanings produce **exactly the same input**. No amount of training
can make a network give different answers to identical inputs. Order carries meaning, and this
representation threw it away.

**Problem 2 — length is fixed.** A dense layer has a fixed number of inputs, set when you build
it. Sentences, audio clips and stock histories come in every length. Padding everything to the
longest possible length is wasteful and still breaks on the first input that is longer.

What we need is a model that reads **one step at a time**, in order, carrying a **memory** of what
it has seen so far — and that works for any length. That is a recurrent neural network.


---
---
# Stage 2 · Watch it unroll

## Idea 2 — One cell, applied at every step

watch StatQuest's
**[Recurrent Neural Networks, Clearly Explained](https://www.youtube.com/watch?v=AsNTP8Kwu80)**
(about 16 minutes). Then run the cell below.


In [ ]:
fig = draw_unrolled_rnn()
plt.show()

Read the picture left to right.

**On the left** is the whole RNN: a single cell with a **loop**. At each time step it takes the
new input `x` *and* its own previous output `h`, and produces a new `h`. That `h` is the
**hidden state** — the network's memory of everything it has read so far.

**On the right** is the same thing **unrolled**: one copy of the cell per time step, with the
memory passed along the chain. It looks like four cells. It is **one** cell, drawn four times.

This is the idea to connect to Lab 3:

| | Lab 3 · CNN | Lab 4 · RNN |
|---|---|---|
| what is shared | one **filter** | one **cell** |
| shared across | every **position** in the image | every **step** in the sequence |
| result | spot a shape anywhere | handle a sequence of any length |
| parameter count depends on image size / sequence length? | **no** | **no** |

Weight sharing, again. Once you see it, you will see it everywhere in deep learning.


---
---
# Stage 3 · Build

## Idea 3 — The RNN cell

The whole cell is one line:

```
h_new = tanh( x · Wx  +  h_prev · Wh  +  b )
```

Two sources of information are combined:

* `x · Wx` — what is arriving **now**, times its weights (exactly like `Layer_Dense`),
* `h_prev · Wh` — what the network **remembers**, times a second set of weights.

Add a bias and squash with **tanh**, which keeps every value between −1 and +1. That squashing
matters: `h` is fed back in at every step, and without a limit it could grow without bound. You
will see why in Idea 5.

The shapes, with `D = 4` features per step and `H = 12` memory units:

| array | shape |
|---|---|
| `x` | `(4,)` — one time step |
| `h_prev`, `h_new` | `(12,)` — the memory, same size every step |
| `Wx` | `(4, 12)` |
| `Wh` | `(12, 12)` — memory to memory |
| `b` | `(12,)` |


### Exercise 1 — one step of an RNN

In [ ]:
def rnn_step(x, h_prev, Wx, Wh, b):
    """One RNN step. Returns the new hidden state."""
    # TODO: tanh( x @ Wx  +  h_prev @ Wh  +  b )
    return ...


# --- self-check ---
x = np.array([1.0, 0.0])
h_prev = np.array([0.5, -0.5])
Wx = np.array([[0.2, 0.4],
               [0.3, -0.1]])
Wh = np.eye(2)                      # memory passed straight through
b = np.array([0.1, 0.0])
# by hand: [0.2 + 0.5 + 0.1,  0.4 - 0.5 + 0.0] = [0.8, -0.1] -> tanh

out = rnn_step(x, h_prev, Wx, Wh, b)
assert out.shape == (2,), f"expected shape (2,), got {out.shape}"
assert np.allclose(out, [0.66403677, -0.09966799]), f"wrong values: {out}"

Wx_, Wh_, b_ = make_rnn_weights()
h1 = rnn_step(np.ones(D), np.zeros(H), Wx_, Wh_, b_)
assert h1.shape == (H,), f"hidden state should be ({H},), got {h1.shape}"
assert np.all(np.abs(h1) < 1), "tanh keeps every value between -1 and 1"
print("Exercise 1 passed")
print("\nh after one step:", h1)

### Exercise 2 — unroll it over a sequence

Start the memory at zeros, then call `rnn_step` once per time step, **feeding each new `h` into
the next step**. Keep every hidden state so you can look at them.


In [ ]:
def rnn_forward(sequence, Wx, Wh, b):
    """Run the RNN over a (T, D) sequence. Returns all hidden states, shape (T, H)."""
    h = np.zeros(Wh.shape[0])          # the memory starts empty
    states = []
    for x in sequence:
        # TODO: update h with one rnn_step, then store it
        h = ...
        states.append(h)
    return np.array(states)


# --- self-check ---
Wx_, Wh_, b_ = make_rnn_weights()
seq = np.random.RandomState(3).randn(5, D)

S = rnn_forward(seq, Wx_, Wh_, b_)
assert S.shape == (5, H), f"5 steps -> (5, {H}) hidden states, got {S.shape}"

# must match three explicit calls chained by hand
h = np.zeros(H)
for x in seq:
    h = rnn_step(x, h, Wx_, Wh_, b_)
assert np.allclose(S[-1], h), "the last state should match a hand-chained loop"

# the memory really is fed forward: change only step 1, the final state must change
seq2 = seq.copy(); seq2[0] += 1.0
assert not np.allclose(rnn_forward(seq2, Wx_, Wh_, b_)[-1], S[-1]), \
    "changing step 1 changed nothing at the end - is h being passed to the next step?"

# the same weights handle ANY length
assert rnn_forward(np.zeros((1, D)), Wx_, Wh_, b_).shape == (1, H)
assert rnn_forward(np.zeros((50, D)), Wx_, Wh_, b_).shape == (50, H)
print("Exercise 2 passed")

print("\nsequence length  -> shape of all hidden states")
for T in [1, 5, 50]:
    print(f"   T = {T:3d}          -> {rnn_forward(np.zeros((T, D)), Wx_, Wh_, b_).shape}")
print("\nSame weights, any length. Problem 2 from Stage 1 is solved.")

---
## Idea 4 — Counting recurrent parameters

The RNN cell has three learnable pieces:

```
Wx : n_in * n_hidden
Wh : n_hidden * n_hidden
b  : n_hidden
```

Look at what is missing: **the sequence length**. A sequence of 5 steps and a sequence of 5,000
use the same weights. Exactly like a CNN's filter cost not depending on image size.

The LSTM and GRU cells are built from the **same block** repeated: each gate is its own
`(Wx, Wh, b)`, the same size as the whole RNN cell.

* **RNN** — 1 block
* **GRU** — 3 blocks (two gates + one candidate)
* **LSTM** — 4 blocks (three gates + one candidate)


### Exercise 3 — count them

In [ ]:
def rnn_params(n_in, n_hidden):
    # TODO: Wx + Wh + b
    return ...


def gru_params(n_in, n_hidden):
    # TODO: 3 blocks
    return ...


def lstm_params(n_in, n_hidden):
    # TODO: 4 blocks
    return ...


# --- self-check ---
assert rnn_params(4, 12) == 204,   "4*12 + 12*12 + 12 = 204"
assert gru_params(4, 12) == 612,   "3 * 204"
assert lstm_params(4, 12) == 816,  "4 * 204"
assert rnn_params(100, 128) == 29312
print("Exercise 3 passed")

print(f"\nfor inputs={D}, hidden={H}:")
print(f"   RNN  {rnn_params(D, H):5,}")
print(f"   GRU  {gru_params(D, H):5,}")
print(f"   LSTM {lstm_params(D, H):5,}")

fig = draw_recurrent_params(D, H)
plt.show()

### Checkpoint 1

1. The hidden size is 12. What is the shape of `h` after step 1? After step 50?
2. You feed a trained RNN a sequence of 10 steps, then one of 1,000 steps. How many parameters
   does it use in each case?
3. Complete the analogy: a CNN shares one ______ across every ______; an RNN shares one ______
   across every ______.


> **Your answers:**
>
> 1. *(write here)*
> 2. *(write here)*
> 3. *(write here)*


---
## Idea 5 — Why a plain RNN forgets

StatQuest makes the key point with a simple example: the memory passes through the **same
weight** at every step. After many steps, the first input has been multiplied by that weight
again and again.

Run the cell to see what repeated multiplication does.


In [ ]:
fig = draw_repeated_multiplication()
plt.show()

Only a weight of exactly 1.0 leaves the signal alone. Anything bigger **explodes**; anything
smaller **vanishes** — and it happens fast. After 20 steps, 0.5 has shrunk the signal a
million-fold.

This is the famous **vanishing gradient** problem (and its twin, the exploding gradient). During
training it means the network cannot learn from things far in the past, because their effect has
been multiplied away. The `tanh` in the cell keeps values from exploding, but it squashes them
further, which only makes the vanishing worse.

### See it in your own RNN

The cell below changes **only the first step** of a sequence and measures how much the **final**
hidden state changes as a result. If the network still remembers step 1, the final state should
move. It uses your `rnn_forward`.


In [ ]:
def influence_of_first_step(final_state_fn, T, n_trials=30):
    """How much does changing step 1 move the final hidden state?"""
    rng = np.random.RandomState(5)
    moves = []
    for _ in range(n_trials):
        noise = rng.randn(T, D) * 0.3
        a = noise.copy(); a[0] = 0; a[0, 0] = 3.0     # class 0 signal at step 1
        b = noise.copy(); b[0] = 0; b[0, 1] = 3.0     # class 1 signal at step 1
        moves.append(np.linalg.norm(final_state_fn(a) - final_state_fn(b)))
    return np.mean(moves)


Wx_, Wh_, b_ = make_rnn_weights()
rnn_final = lambda s: rnn_forward(s, Wx_, Wh_, b_)[-1]

lengths = [1, 2, 5, 10, 20, 40, 80]
rnn_influence = [influence_of_first_step(rnn_final, T) for T in lengths]

for T, v in zip(lengths, rnn_influence):
    print(f"T = {T:3d}   step-1 influence on final state = {v:.5f}")

plt.figure(figsize=(7, 3.8))
plt.semilogy(lengths, np.maximum(rnn_influence, 1e-12), "o-", lw=2,
             color="#C0392B", label="RNN")
plt.xlabel("sequence length T"); plt.ylabel("influence of step 1 (log scale)")
plt.title("A plain RNN forgets the first step"); plt.grid(alpha=.3, which="both")
plt.legend(); plt.show()

By 20 steps the first input has almost no effect on the final state; by 40 it has effectively
none. The RNN has not stored step 1 badly — it has **lost it entirely**.

That is the problem LSTMs were invented to solve, in 1997.


---
## Idea 6 — The LSTM: two kinds of memory, three stages

Now watch StatQuest's
**[Long Short-Term Memory (LSTM), Clearly Explained](https://www.youtube.com/watch?v=YCzL96nL7j0)**
(about 21 minutes). This section follows its structure exactly.

### Two kinds of memory

An LSTM keeps **two** memories instead of one:

* **long-term memory** — the **cell state** `c`. It runs along the top of the cell like a
  conveyor belt, and is changed only by gentle multiplication and addition.
* **short-term memory** — the **hidden state** `h`, the same thing a plain RNN has.

Run the cell to see how they fit together.


In [ ]:
fig = draw_lstm_cell()
plt.show()

### The three stages

Each stage is controlled by a **gate** — a sigmoid, so its output is between 0 and 1, which you
can read as a **percentage**. Every gate is built exactly like an RNN cell: `x · Wx + h · Wh + b`.

**Stage 1 · How much to remember** (red) — the **forget gate** `f`

```
f = sigmoid( x · Wf_x + h_prev · Wf_h + bf )
```

`f` near 1 means *keep* the long-term memory; near 0 means *wipe it*. StatQuest calls this
"the percent of long-term memory to remember".

**Stage 2 · Update long-term memory** (green) — the **input gate** `i` and the **candidate** `g`

```
g = tanh(    ... )      what could be added
i = sigmoid( ... )      what percentage of it to actually add
c = f * c_prev  +  i * g
```

**Stage 3 · Update short-term memory** (blue) — the **output gate** `o`

```
o = sigmoid( ... )
h = o * tanh(c)
```

### Why this fixes forgetting

Look at the long-term memory update: `c = f * c_prev + i * g`.

There is no weight matrix and no `tanh` between `c_prev` and `c` — just a multiply by `f` and an
addition. If the forget gate learns `f ≈ 1`, the memory rides the conveyor belt almost untouched,
step after step. Go back to the repeated-multiplication plot: **the line that stays flat is
w = 1.0**. The forget gate lets the network choose to be that line.


### Exercise 4 — one step of an LSTM

`W` is a dict: `W["f"]`, `W["i"]`, `W["g"]`, `W["o"]`, each a `(Wx, Wh, b)` tuple — the same
three pieces as an RNN cell. A helper computes `x · Wx + h · Wh + b` for any gate.


In [ ]:
def gate_input(x, h_prev, params):
    """x @ Wx + h_prev @ Wh + b for one gate. Same as the inside of an RNN cell."""
    Wx, Wh, b = params
    return x @ Wx + h_prev @ Wh + b


def lstm_step(x, h_prev, c_prev, W):
    """One LSTM step. Returns (h_new, c_new)."""
    # TODO Stage 1: forget gate  (sigmoid)
    f = ...

    # TODO Stage 2: input gate (sigmoid) and candidate memory (tanh)
    i = ...
    g = ...

    # TODO Stage 2: new long-term memory
    c = ...

    # TODO Stage 3: output gate (sigmoid) and new short-term memory
    o = ...
    h = ...

    return h, c


# --- self-check ---
def fixed_gates(f_bias, i_bias, o_bias, g_bias=0.0):
    """All weights zero, so each gate is just sigmoid/tanh of its bias."""
    W = {}
    for name, bias in [("f", f_bias), ("i", i_bias), ("g", g_bias), ("o", o_bias)]:
        W[name] = (np.zeros((D, H)), np.zeros((H, H)), np.full(H, float(bias)))
    return W

x0, h0 = np.ones(D), np.zeros(H)
c0 = np.linspace(-1, 1, H)

# forget gate fully open, input gate shut: the memory rides the belt untouched
h, c = lstm_step(x0, h0, c0, fixed_gates(f_bias=50, i_bias=-50, o_bias=50))
assert h.shape == (H,) and c.shape == (H,), "both memories should have shape (H,)"
assert np.allclose(c, c0), "f=1, i=0 should leave the long-term memory unchanged"
assert np.allclose(h, np.tanh(c)), "with o=1, h should be tanh of the NEW c"

# forget everything, add nothing: memory wiped
_, c = lstm_step(x0, h0, c0, fixed_gates(f_bias=-50, i_bias=-50, o_bias=50))
assert np.allclose(c, 0), "f=0, i=0 should wipe the long-term memory"

# forget everything, write the candidate
_, c = lstm_step(x0, h0, c0, fixed_gates(f_bias=-50, i_bias=50, o_bias=50, g_bias=0.5))
assert np.allclose(c, np.tanh(0.5)), "f=0, i=1 should replace the memory with g = tanh(0.5)"

# real weights: shapes and ranges
h, c = lstm_step(np.random.RandomState(1).randn(D), h0, np.zeros(H), make_lstm_weights())
assert np.all(np.abs(h) < 1), "h = o * tanh(c) must stay between -1 and 1"
print("Exercise 4 passed")

Now give the LSTM the same memory test as the RNN. The cell uses your `lstm_step`.

In [ ]:
def lstm_forward(sequence, W):
    """Run the LSTM over a (T, D) sequence. Returns the final (h, c)."""
    h, c = np.zeros(H), np.zeros(H)
    for x in sequence:
        h, c = lstm_step(x, h, c, W)
    return h, c


W_lstm = make_lstm_weights()
lstm_final = lambda s: lstm_forward(s, W_lstm)[0]
lstm_influence = [influence_of_first_step(lstm_final, T) for T in lengths]

print(f"{'T':>4}   {'RNN':>10}   {'LSTM':>10}")
for T, r, l in zip(lengths, rnn_influence, lstm_influence):
    print(f"{T:4d}   {r:10.5f}   {l:10.5f}")

plt.figure(figsize=(7, 3.8))
plt.semilogy(lengths, np.maximum(rnn_influence, 1e-12), "o-", lw=2,
             color="#C0392B", label="RNN")
plt.semilogy(lengths, lstm_influence, "s-", lw=2, color="#1565C0", label="LSTM")
plt.xlabel("sequence length T"); plt.ylabel("influence of step 1 (log scale)")
plt.title("The LSTM keeps the first step"); plt.grid(alpha=.3, which="both")
plt.legend(); plt.show()

The RNN's line dives toward zero. The LSTM's stays level: step 1 still moves its final state
after 80 steps.

This LSTM has **frozen, untrained** weights — the same kind of random weights as the RNN. The only
difference is that its forget gate is set to *keep* (bias +4, so `f ≈ 0.98`). The architecture
alone makes the difference.


---
## Idea 7 — The GRU: a lighter LSTM

StatQuest does not cover GRUs. Read Michael Phi's
**[Illustrated Guide to LSTM's and GRU's](https://towardsdatascience.com/illustrated-guide-to-lstms-and-gru-sa-step-by-step-explanation-44e9eb85bf21)**
(the GRU half; a video version is linked in the article).

The **Gated Recurrent Unit** (2014) keeps the good idea of the LSTM — gates that decide what to
keep — with less machinery:

* **one memory** instead of two: there is no separate cell state `c`, only `h`,
* **two gates** instead of three:
  * the **update gate** `z` decides how much of the old memory to keep versus replace — it does
    the job of the LSTM's forget *and* input gates together,
  * the **reset gate** `r` decides how much of the past to use when proposing new memory.

```
z = sigmoid( ... )                      update gate
r = sigmoid( ... )                      reset gate
h_cand = tanh( x · Wx + (r * h_prev) · Wh + b )
h = (1 - z) * h_prev  +  z * h_cand
```

The last line has the same shape as the LSTM's conveyor belt: keep some old, add some new.

### Side by side

| | RNN | GRU | LSTM |
|---|---|---|---|
| memories | `h` | `h` | `c` (long-term) + `h` (short-term) |
| gates | none | 2 (update, reset) | 3 (forget, input, output) |
| blocks of parameters | 1 | 3 | 4 |
| parameters here (4 in, 12 hidden) | 204 | 612 | 816 |
| long memory | poor | good | good |
| typical use | short sequences, teaching | a lighter default; faster to train | the long-standing default |

In practice LSTM and GRU often perform similarly. GRU is smaller and faster; LSTM has the separate
long-term memory. It is common to try both.


### Checkpoint 2

1. In an LSTM, which gate is StatQuest's "percent to remember"? What value means "keep
   everything"?
2. The RNN's memory vanished but the LSTM's did not. Point to the one line of the LSTM that makes
   the difference, and explain why.
3. Give one reason you might choose a GRU over an LSTM.


> **Your answers:**
>
> 1. *(write here)*
> 2. *(write here)*
> 3. *(write here)*


---
---
# Stage 4 · Prove it

# Assessment

**Marks: 1** (scored out of 10 below, then scaled).

### The task

Every sequence is **T** steps long, with 4 features per step. **Step 1 carries the answer** — a
spike in feature 0, 1 or 2 tells you the class. **Every later step is noise.** To classify
correctly, a model has to carry step 1 all the way to the end, through T − 1 steps of
distraction.

Which model can remember, and for how long?

### What is frozen, and what is trained

As in Lab 3, we have not yet built backpropagation through a recurrent network (it is called
*backpropagation through time*). So the recurrent cell is **frozen**, and we train only a dense
head on its **final hidden state** — the transfer-learning pattern again.

The LSTM's gate biases are set by hand: forget gate **+4** (keep), input gate **−2** (mostly
closed to noise). Setting the forget-gate bias to a positive value is a real, widely used
practice, so the cell starts out remembering. Think of it as Lab 3's Sobel filters: hand-set, but
the kind of thing training would find.


In [ ]:
# --- Assessment setup: DO NOT MODIFY ---

fig = draw_lab4_pipeline()
plt.show()

X_demo, y_demo = make_sequences(1, T=12, seed=7)
fig, axes = plt.subplots(1, 3, figsize=(11, 2.6))
for k in range(3):
    axes[k].imshow(X_demo[k].T, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
    axes[k].set_title(f"class {k}: spike in feature {k} at step 1", fontsize=9.5)
    axes[k].set_xlabel("time step"); axes[k].set_yticks(range(D))
    axes[k].set_xticks([0, 5, 11]); axes[k].set_xticklabels([1, 6, 12])
axes[0].set_ylabel("feature")
plt.tight_layout(); plt.show()

RNN_W = make_rnn_weights()
LSTM_W = make_lstm_weights(forget_bias=4.0, input_bias=-2.0)
print("frozen RNN and LSTM weights ready")

### Q1 — Read out the final memory *(3 marks)*

Write `final_hidden_states`, which runs every sequence through a model and returns the **final**
hidden state of each, stacked into a matrix. Then train a head on each model at T = 10.


In [ ]:
def final_hidden_states(X, model):
    """X has shape (n, T, D). Returns (n, H): the last h of each sequence."""
    out = []
    for seq in X:
        if model == "rnn":
            # TODO: the LAST row of rnn_forward (use RNN_W)
            out.append(...)
        else:
            # TODO: the final h from lstm_forward (use LSTM_W)
            out.append(...)
    return np.array(out)


X_tr, y_tr = make_sequences(40, T=10, seed=1)
X_te, y_te = make_sequences(40, T=10, seed=2)

results_q1 = {}
for model in ["rnn", "lstm"]:
    # TODO: final states for train and test, train a head, record test accuracy
    F_tr = ...
    F_te = ...
    head = ...
    results_q1[model] = accuracy(head, F_te, y_te)

print("T = 10, test accuracy")
for m, a in results_q1.items():
    print(f"   {m.upper():5s} {a:.3f}")

# --- self-check ---
assert final_hidden_states(X_tr[:3], "rnn").shape == (3, H)
assert final_hidden_states(X_tr[:3], "lstm").shape == (3, H)
assert results_q1["lstm"] > 0.9, "the LSTM should still remember step 1 after 10 steps"
assert results_q1["rnn"] < 0.5, "the RNN should have forgotten step 1 by 10 steps"
print("\nQ1 passed")

### Q2 — How long can each one remember? *(2 marks)*

Repeat Q1 across sequence lengths. Takes about 10 seconds.


In [ ]:
lengths_q2 = [2, 5, 10, 20, 40, 80]
table = []

for T in lengths_q2:
    Xtr, ytr = make_sequences(40, T=T, seed=1)
    Xte, yte = make_sequences(40, T=T, seed=2)
    row = [T]
    for model in ["rnn", "lstm"]:
        # TODO: same three steps as Q1, then append the test accuracy to row
        ...
    table.append(row)

print(f"{'T':>4}   {'RNN':>6}   {'LSTM':>6}")
for T, r, l in table:
    print(f"{T:4d}   {r:6.3f}   {l:6.3f}")

plt.figure(figsize=(7, 3.8))
plt.plot(lengths_q2, [r[1] for r in table], "o-", lw=2, color="#C0392B", label="RNN")
plt.plot(lengths_q2, [r[2] for r in table], "s-", lw=2, color="#1565C0", label="LSTM")
plt.axhline(1/3, color="grey", ls="--", lw=1, label="chance")
plt.xscale("log"); plt.xticks(lengths_q2, lengths_q2)
plt.xlabel("sequence length T"); plt.ylabel("test accuracy"); plt.ylim(0, 1.05)
plt.title("Remembering step 1"); plt.legend(); plt.grid(alpha=.3); plt.show()

# --- self-check ---
assert len(table) == len(lengths_q2) and all(len(r) == 3 for r in table)
assert table[4][1] < 0.5, "at T=40 the RNN should be near chance"
assert table[4][2] > 0.85, "at T=40 the LSTM should still remember"
print("Q2 passed")

### Q3 — Turn the forget-gate dial *(2 marks)*

The forget gate is "the percent to remember". Its bias sets that percent: `sigmoid(bias)`.
Rebuild the LSTM with different forget biases and measure its memory at T = 20.


In [ ]:
X20_tr, y20_tr = make_sequences(40, T=20, seed=1)
X20_te, y20_te = make_sequences(40, T=20, seed=2)

dial = []
for fb in [-2.0, 0.0, 2.0, 4.0]:
    # TODO: rebuild LSTM_W with this forget bias (keep input_bias=-2.0)
    LSTM_W = ...
    # TODO: final states, train a head, test accuracy
    acc = ...
    dial.append((fb, sigmoid(fb), acc))

LSTM_W = make_lstm_weights(forget_bias=4.0, input_bias=-2.0)   # restore

print(f"{'forget bias':>11}   {'% kept per step':>15}   {'test acc (T=20)':>15}")
for fb, keep, a in dial:
    print(f"{fb:+11.1f}   {keep:15.2f}   {a:15.3f}")

# --- self-check ---
assert len(dial) == 4
assert dial[3][2] > dial[1][2] + 0.4, "a forget bias of +4 should remember far better than 0"
assert dial[3][2] == max(d[2] for d in dial), "+4 should be the best setting here"
print("\nQ3 passed")

### Q4 — Short answers *(3 marks)*

1. At T = 2 the RNN is perfect, but by T = 10 it scores about 0.33. What does 0.33 mean for a
   3-class problem, and *why* has the RNN lost step 1? Use the idea from Idea 5.
2. The forget gate keeps a fraction `sigmoid(bias)` of the memory **each step**, and that
   compounds. Calculate `0.88 ** 20` and `0.98 ** 20`, and use them to explain your Q3 table.
3. At T = 80 even the LSTM drops noticeably. Does an LSTM have perfect memory? What does your Q2
   result suggest about very long sequences?


> **Your answers:**
>
> 1. *(write here)*
> 2. *(write here)*
> 3. *(write here)*


---

## What you built today

A recurrent network and an LSTM, from scratch, and a demonstration of the difference: with
identical frozen weights, **the RNN forgot step 1 within about 5 steps; the LSTM still held it
after 40.**

| Piece | You wrote | Idea |
|---|---|---|
| RNN cell | `rnn_step` | new memory = tanh(input + old memory) |
| Unrolling | `rnn_forward` | one cell, every time step — weight sharing across time |
| Counting | `rnn_params`, `gru_params`, `lstm_params` | 1, 3 and 4 blocks; independent of length |
| LSTM cell | `lstm_step` | two memories, three stages, an additive conveyor belt |
| Readout | `final_hidden_states` | frozen recurrent base + trained head |

### What was simplified

* **The recurrent weights were frozen.** Training them needs *backpropagation through time* —
  the chain rule from Lab 2 applied back along the unrolled chain. Frameworks do this for you in
  Labs 7–8.
* **The gate biases were set by hand.** A trained LSTM learns them.
* **The GRU was only described**, not implemented. Its code is a small variation on `lstm_step`
  if you want to try.
* **The task was synthetic.** Real uses — text, speech, sensor data — follow the same pattern.


### Submission

1. Create a new repository on GitHub for this course and name it `arti402`.
2. Rename this file to `arti402_Lab4_<YourID>.ipynb` and upload the completed notebook, with all
   outputs visible, to your repository.
